# GENE7 cohort-specific axes of intra-cohort variability

**Question.** Not "where does each cohort sit in the morphospace" but: *along which directions does
each cohort vary internally, and are those directions cohort-specific?*

**Design** (see `cohort_axes.py` for the full rationale):

- A **GENE7-native 10-component PCA** on `z_mu_b_*`, used only as a denoised coordinate system. The
  global PCs themselves are not of interest. GENE7-native matters: a reference-fit basis is
  wildtype-dominated and would pre-filter out the perturbation directions we are hunting.
- **Unwhitened.** Whitening was tried first (to stop cohort axes drifting toward the dominant global
  direction) but rejected: `z_mu_b`'s spectrum is steep — PC1 holds 47.8% while PC8-10 hold ~0.1%
  each — so whitening inflates a near-degenerate tail to parity with real signal. It was verified
  to change **no** conclusion (see the sensitivity check below), while diluting the dominant
  severity axis (leading-axis variance 0.45 whitened vs 0.64 unwhitened).
- **Centered per cohort**, so we recover intra-cohort *variability* rather than cohort *position*.
- **Cohort = target x temperature x timepoint**: 48 cohorts, 9-12 wells each.
- Top **2-3 axes** only, with bootstrap eigenvalue CIs, subsample stability, and a **permutation
  null** from random same-arm pseudo-cohorts of the same size.

> **The null is the point.** At n~12 in 10 dimensions an arbitrary group of embryos still yields a
> leading axis with a large variance share. Without the null, any structure found here is
> uninterpretable.

Regenerate with:

```bash
PYTHONPATH=/net/trapnell/vol1/home/nlammers/projects/repositories/morphseq/src:. \
  /net/trapnell/vol1/home/nlammers/micromamba/envs/points-ml/bin/python run_cohort_axes.py
```

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parents[2] / "src"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import cohort_axes as ca
import plotting as pl

pio.renderers.default = "notebook"
DATA = HERE / "data"

# shared figure config -- see gene7_config.py
import gene7_config as cfg
FIGS = cfg.figure_dir("cohort_axes")
print(f"figures -> {FIGS}")

scores     = pd.read_csv(DATA / "gene7_global_scores.csv")
variance   = pd.read_csv(DATA / "gene7_global_variance.csv")
sizes      = pd.read_csv(DATA / "cohort_sizes.csv")
summary    = pd.read_csv(DATA / "cohort_summary.csv")
loadings   = pd.read_csv(DATA / "cohort_loadings.csv")
similarity = pd.read_csv(DATA / "cohort_similarity.csv", index_col=0)
null       = pd.read_csv(DATA / "cohort_null.csv")
strips     = pd.read_csv(DATA / "image_strips.csv")

print(f"{len(scores)} wells | {len(sizes)} cohorts | sizes {sizes.n_wells.min()}-{sizes.n_wells.max()}")

## The 10D coordinate system

In [ ]:
display(variance.round(4))
GC = [c for c in ca.GLOBAL_COLUMNS if c in scores.columns]
print("per-axis variance (unwhitened):", np.round(scores[GC].var(ddof=1).to_numpy(), 3))
print("\nThe spectrum is steep: components 8-10 carry ~0.1% each, so the 10D space is effectively ~7D.")
print("This steepness is exactly why whitening was rejected -- it would scale those up to parity.")

## Cohort spectra against the null

`axis_k_var_ratio` is the share of the cohort's own variance along its k-th axis.
`axis_k_null_pctile` is where that sits in the distribution of random same-arm pseudo-cohorts.

In [ ]:
print("=== permutation null (n=12 pseudo-cohorts, 300 draws) ===")
display(null[["axis_1", "axis_2", "axis_3"]].describe().round(3))

print("=== real cohorts ===")
for column in ["axis_1_var_ratio", "axis_2_var_ratio"]:
    print(f"  {column:22s} mean={summary[column].mean():.3f}  max={summary[column].max():.3f}")
print(f"\ncohorts with axis_1 above the 95th null percentile: "
      f"{(summary.axis_1_null_pctile > 95).sum()} / {len(summary)}")

In [ ]:
fig = px.histogram(null, x="axis_1", nbins=40, opacity=0.75,
                   labels={"axis_1": "leading-axis variance ratio"},
                   title="Real cohorts vs the no-structure null (leading axis)")
fig.data[0].name = "null (random pseudo-cohorts)"
fig.add_histogram(x=summary.axis_1_var_ratio, nbinsx=40, opacity=0.75,
                  name="real cohorts", marker_color="crimson")
fig.update_layout(width=900, height=480, font=pl.FONT, plot_bgcolor="white", barmode="overlay",
                  legend=dict(x=0.02, y=0.98))
pl.save_figure(fig, FIGS / "cohort_axis1_vs_null")
fig.show()

The two distributions sit on top of each other. **The amount of variance concentration in a real
cohort is fully explained by having 12 samples in 10 dimensions.**

The axes are nonetheless *stable* (mean subsample agreement ~0.90 for axis 1), which is a different
question from whether their variance share is remarkable. So the meaningful test is whether the axis
**directions** are cohort-specific.

In [ ]:
fig = px.scatter(summary, x="axis_1_var_ratio", y="axis_1_stability",
                 color="temperature", symbol="target", size="n_wells",
                 color_continuous_scale="RdBu_r", hover_name="cohort",
                 labels={"axis_1_var_ratio": "leading-axis variance ratio",
                         "axis_1_stability": "subsample stability (|cos|)"},
                 title="Leading axis: variance share vs directional stability")
fig.add_vline(x=null.axis_1.quantile(0.95), line_dash="dash", line_color="grey",
              annotation_text="95th null pctile")
fig.update_layout(width=950, height=620, font=pl.FONT, plot_bgcolor="white")
pl.save_figure(fig, FIGS / "cohort_axis1_variance_vs_stability")
fig.show()

## Are the axis directions cohort-specific?

Compared by **principal angles between top-2 subspaces**, not PC1-to-PC1 pairing: at this n the
eigenvalue gaps are small, so component *ordering* flips readily even when the *subspace* is stable.
Similarity is the mean cosine of the two principal angles (1 = same plane, 0 = orthogonal).

In [ ]:
labels = list(similarity.index)
values = similarity.to_numpy()
upper = np.triu_indices(len(labels), 1)

target = [lbl.split(" | ")[0] for lbl in labels]
temperature = [lbl.split(" | ")[1] for lbl in labels]
timepoint = [lbl.split(" | ")[2] for lbl in labels]

def contrast(key):
    same = [values[i, j] for i, j in zip(*upper) if key[i] == key[j]]
    diff = [values[i, j] for i, j in zip(*upper) if key[i] != key[j]]
    return np.mean(same), np.mean(diff)

print(f"all cohort pairs                     mean = {values[upper].mean():.3f}")
rng = np.random.RandomState(0)
randoms = []
for _ in range(500):
    a = np.linalg.qr(rng.randn(10, 2))[0].T
    b = np.linalg.qr(rng.randn(10, 2))[0].T
    randoms.append(np.cos(np.radians(ca.principal_angles(a, b, k=2))).mean())
print(f"random 2-planes in 10D               mean = {np.mean(randoms):.3f}   <- the floor")
print()
for name, key in [("same target", target), ("same temperature", temperature),
                  ("same timepoint", timepoint)]:
    same, diff = contrast(key)
    print(f"  {name:18s} same={same:.3f}  diff={diff:.3f}  delta={same - diff:+.3f}")

**This is the result.**

- Cohort subspaces (~0.42) sit barely above the random-plane floor (~0.38).
- **`same target` delta is ~0.000** — cohorts sharing a crispant target agree no more than cohorts
  that do not. There is no detectable genotype-specific axis of intra-cohort variability at n=12.
- **Temperature is the only signal** (delta ~ +0.09): cohorts reared at the same temperature vary
  along more similar directions.

In [ ]:
order = np.argsort([f"{t} {p}" for t, p in zip(temperature, target)])
fig = px.imshow(values[np.ix_(order, order)],
                x=[labels[i] for i in order], y=[labels[i] for i in order],
                color_continuous_scale="Viridis", zmin=0, zmax=1,
                title="Pairwise top-2 subspace similarity (sorted by temperature, then target)")
fig.update_layout(width=980, height=900, font=dict(size=8))
pl.save_figure(fig, FIGS / "cohort_subspace_similarity")
fig.show()

## Does pooling recover a genotype signal?

The n=12 cohort is thin by construction. If a genotype-specific axis exists but is merely
under-powered, pooling should surface it.

In [ ]:
from sklearn.decomposition import PCA

def subspace(frame, k=3):
    return PCA(n_components=k, random_state=42).fit(frame[GC].to_numpy(float))

def pair_similarity(a, b, k=2):
    return float(np.cos(np.radians(ca.principal_angles(a.components_, b.components_, k=k))).mean())

print("=== pooled over timepoint: target x temperature (n~35) ===")
pooled = {keys: subspace(group) for keys, group in scores.groupby(["perturbation_group", "temperature"])}
keys = list(pooled)
matrix = np.array([[pair_similarity(pooled[a], pooled[b]) for b in keys] for a in keys])
up = np.triu_indices(len(keys), 1)
for name, index in [("same target", 0), ("same temperature", 1)]:
    same = [matrix[i, j] for i, j in zip(*up) if keys[i][index] == keys[j][index]]
    diff = [matrix[i, j] for i, j in zip(*up) if keys[i][index] != keys[j][index]]
    print(f"  {name:18s} same={np.mean(same):.3f}  diff={np.mean(diff):.3f}  "
          f"delta={np.mean(same) - np.mean(diff):+.3f}")

print("\n=== pooled to target only (n~142) ===")
for name, group in scores.groupby("perturbation_group"):
    spectrum = subspace(group).explained_variance_ratio_
    print(f"  {name:14s} n={len(group):4d}  top-3 variance ratio = {np.round(spectrum, 3)}")

Pooling **strengthens temperature** (delta rises to ~+0.26 at n~35) while `same target` stays flat
or slightly negative. At n~142 per target the spectra are near-identical across genotypes
(0.14-0.18 for the first three components) — no genotype concentrates variance differently.

The negative result is therefore **robust across three sample sizes**, not an artifact of thin
cohorts.

## Image strips along the cohort axes

Embryos sampled evenly along each cohort's own top-2 axes, ordered low to high, so the axes can be
read visually. Numbers above each tile are the coordinate along that axis.

> **Caveat.** These are the CURRENT pipeline's snips — a degraded raster (smaller, ~4x more
> saturated) than the build that produced the legacy latents. Fine for judging shape; they are not
> the pixels the latents were computed from.

Given the null result above, expect these to read as generic embryo-to-embryo variation (yolk
extension, body curvature, mounting orientation) rather than a crisp genotype phenotype.

In [ ]:
strips["temperature"] = [lbl.split(" | ")[1] for lbl in strips.cohort]
strips["target"] = [lbl.split(" | ")[0] for lbl in strips.cohort]
print(f"{len(strips)} tiles, {strips.image_exists.sum()} resolved | {strips.cohort.nunique()} cohorts")
print("tiles per cohort-axis:", sorted(strips.groupby(['cohort','axis']).size().unique()))

# Render ALL 48 cohorts x 2 axes to figures/strips/ (not shown inline -- 96 figures).
import matplotlib.pyplot as plt
strip_dir = FIGS / "strips"; strip_dir.mkdir(exist_ok=True)
count = 0
for cohort in sorted(strips.cohort.unique()):
    for axis in ("axis_1", "axis_2"):
        sub = strips[(strips.cohort == cohort) & (strips.axis == axis)]
        if sub.empty:
            continue
        figure = pl.image_strip_figure(sub, coord_column=axis, title=f"{cohort}  -  {axis}")
        name = cohort.replace(" | ", "_").replace(",", "-")
        figure.savefig(strip_dir / f"{name}_{axis}.png", dpi=100, bbox_inches="tight")
        plt.close(figure); count += 1
print(f"rendered {count} strips to {strip_dir}")

In [ ]:
# Strips for every cohort are rendered to figures/strips/ by the cell below.
# Shown inline: the widest-spread cohorts (clearest severity gradient) and their Control counterparts.
import matplotlib.pyplot as plt

show = ["atf6 | 34C | 36hpf", "ctcf | 28C | 36hpf",
        "Control | 34C | 36hpf", "Control | 28C | 36hpf"]
for cohort in show:
    sub = strips[(strips.cohort == cohort) & (strips.axis == "axis_1")]
    if sub.empty:
        continue
    figure = pl.image_strip_figure(sub, coord_column="axis_1", title=f"{cohort}  -  axis_1")
    display(figure)
    plt.close(figure)

### A temperature contrast at matched genotype and timepoint

Temperature was the one signal that survived, so this is the comparison worth looking at.

In [ ]:
import matplotlib.pyplot as plt

for cohort in ["Control | 24C | 36hpf", "Control | 35C | 36hpf"]:
    sub = strips[(strips.cohort == cohort) & (strips.axis == "axis_1")]
    if sub.empty:
        print(f"(no strip for {cohort})")
        continue
    figure = pl.image_strip_figure(sub, coord_column="axis_1", title=f"{cohort}  -  axis_1")
    figure.savefig(FIGS / f"strip_temp_{cohort.replace(' | ', '_')}.png", dpi=110,
                   bbox_inches="tight")
    display(figure)
    plt.close(figure)

## Sensitivity check: was it the whitening?

Whitening inflates near-degenerate directions, which at n~12 is exactly where noise lives — a real
worry that it manufactured the null rather than revealed one. Re-run across configurations:

In [ ]:
from sklearn.decomposition import PCA
import morph_pca_spline as mps
from morphseq_integration import build_master_table

g7 = mps.gene7_latents_from_master(build_master_table().query("has_morph").copy())

def configuration(whiten, n_components):
    basis = ca.fit_global_basis(g7, n_components=n_components, whiten=whiten)
    cohorts = ca.fit_all_cohorts(basis, bootstrap=False, stability=False)
    matrix = ca.cohort_similarity_matrix(cohorts, k=2).to_numpy()
    up = np.triu_indices(len(cohorts), 1)
    def delta(key):
        same = [matrix[i, j] for i, j in zip(*up) if key[i] == key[j]]
        diff = [matrix[i, j] for i, j in zip(*up) if key[i] != key[j]]
        return np.mean(same) - np.mean(diff)
    return {
        "config": f"{'whitened' if whiten else 'unwhitened'} {n_components}D",
        "pair_similarity": matrix[up].mean(),
        "delta_target": delta([c.keys[0] for c in cohorts]),
        "delta_temperature": delta([c.keys[1] for c in cohorts]),
        "mean_axis1_var": np.mean([c.variance_ratio[0] for c in cohorts]),
    }

display(pd.DataFrame([configuration(w, n) for w, n in
                      [(True, 10), (False, 10), (True, 5), (False, 5), (False, 3)]]).round(3))

**Whitening was not the culprit.** `delta_target` stays at ~0.000 in every configuration, whitened or
not, at 3/5/10 components. The negative genotype result is robust to both choices.

Whitening *was* the wrong call for a different reason, and the table shows it: unwhitened, the leading
axis captures 0.64 of cohort variance versus 0.45 whitened. Whitening was diluting the single dominant
severity axis by promoting the degenerate tail. The cached artifacts are now unwhitened.

## Where genotype *does* show up: heterogeneity, not direction

Cohorts do not differ in the *direction* they vary. But they differ in *how far* they spread along the
shared severity axis — and there the crispants separate from Control.

In [ ]:
strip_axis1 = strips[strips.axis == "axis_1"].copy()
strip_axis1["temp"] = [int(c.split(" | ")[1].rstrip("C")) for c in strip_axis1.cohort]
spread = (strip_axis1.groupby(["temp", "target"])["axis_1"]
          .agg(lambda v: v.max() - v.min()).unstack())
print("axis-1 coordinate spread (max - min) by temperature x target:")
display(spread.round(2))
print("Control is the NARROWEST spread at 3 of 4 temperatures -- consistent with mosaic F0")
print("crispants producing a wider range of severities within a single well group.")

## Read-out

1. **Within-cohort phenotypic variability is real and large.** The image strips show it directly:
   at stressed conditions, cohorts run from well-extended straight embryos to short, curled,
   malformed ones. A single **severity axis** dominates, capturing ~0.64 of each cohort's variance.

2. **That axis is shared, not cohort-specific.** Cohorts sharing a crispant target vary along no more
   similar directions than cohorts that do not (`delta_target` ~ 0.000, robust to whitening and to
   3/5/10 components, and at n=12, n~35, n~142). So the earlier phrasing "no structure" was wrong —
   the correct statement is that all cohorts vary along essentially the *same* direction.

3. **The *amount* of variance concentration is a small-n artifact.** Real cohorts and random same-arm
   pseudo-cohorts have indistinguishable leading-axis variance ratios, and 0/48 cohorts clear the
   95th null percentile. Any per-cohort eigenvalue read without the null would be over-interpreted.

4. **Genotype shows up as heterogeneity, not direction.** Control has the narrowest spread along the
   severity axis at 3 of 4 temperatures; the three crispants are wider. That is the expected
   signature of mosaic F0 injections, and it is a *scale* effect the subspace comparison is blind to
   by construction.

5. **Temperature modulates the shared axis** (+0.05 to +0.09 subspace delta, rising to +0.26 pooled),
   and the coordinate spread is widest at 28C.

**Caveats.** The 10D space is effectively ~7D. GENE7 has no per-embryo morphological stage
(`predicted_stage_hpf` is nominal collection age), so residual within-cohort stage variation cannot be
regressed out — some of the severity axis may be developmental delay rather than malformation, which
these snips alone cannot separate. And `z_mu_b` is only *nominally* nuisance-free: if orientation
signal appears in the loadings, the VAE's disentanglement is imperfect, not a leak of `z_mu_n_*`
(which were never included).